In [3]:
import os
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds
import subprocess
import tempfile

def xyz_to_smiles_obabel(xyz_file):
    """
    Конвертирует XYZ в SMILES через Open Babel
    """
    try:
        result = subprocess.run(
            ['obabel', xyz_file, '-osmi'],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0 and result.stdout.strip():
            smiles = result.stdout.strip().split()[0]  # Первый элемент - SMILES
            return smiles
        return None
    except Exception as e:
        return None

def xyz_to_smiles_rdkit(xyz_file):
    """
    Конвертирует XYZ в SMILES через RDKit
    """
    try:
        mol = Chem.MolFromXYZFile(xyz_file)
        if mol is None:
            return None
        
        # Пробуем определить связи
        rdDetermineBonds.DetermineBonds(mol, charge=0)
        smiles = Chem.MolToSmiles(mol)
        return smiles
    except:
        return None

def xyz_to_smiles(xyz_file, method='both'):
    """
    Пробует разные методы конвертации
    """
    # Сначала пробуем RDKit
    smiles = xyz_to_smiles_rdkit(xyz_file)
    if smiles:
        return smiles
    
    # Если RDKit не сработал, пробуем Open Babel
    if method in ['both', 'obabel']:
        smiles = xyz_to_smiles_obabel(xyz_file)
        if smiles:
            return smiles
    
    return None

def process_xyz_folder(folder_path, method='both'):
    """
    Обрабатывает все XYZ файлы в папке
    
    Args:
        folder_path: путь к папке с XYZ файлами
        method: 'rdkit', 'obabel', или 'both'
    """
    result = {}
    folder = Path(folder_path)
    xyz_files = sorted(folder.glob("*.xyz"))
    
    if not xyz_files:
        print(f"В папке {folder_path} не найдено XYZ файлов")
        return result
    
    print(f"Найдено {len(xyz_files)} XYZ файлов")
    print("Начинаю обработку...\n")
    
    success_count = 0
    fail_count = 0
    failed_files = []
    
    for i, xyz_file in enumerate(xyz_files, 1):
        filename = xyz_file.stem
        smiles = xyz_to_smiles(str(xyz_file), method=method)
        
        if smiles:
            result[filename] = smiles
            success_count += 1
            if i % 50 == 0:
                print(f"Обработано {i}/{len(xyz_files)} файлов... (Успешно: {success_count})")
        else:
            fail_count += 1
            failed_files.append(filename)
    
    print(f"\n{'='*50}")
    print(f"Готово! Успешно: {success_count}, Ошибок: {fail_count}")
    print(f"{'='*50}\n")
    
    # Сохраняем список неудачных файлов
    if failed_files:
        with open("failed_files.txt", 'w') as f:
            for fname in failed_files:
                f.write(f"{fname}\n")
        print(f"Список неудачных файлов сохранён в failed_files.txt")
    
    return result

# Проверяем наличие Open Babel
def check_obabel():
    try:
        subprocess.run(['obabel', '-V'], capture_output=True, timeout=2)
        print("✓ Open Babel установлен\n")
        return True
    except:
        print("✗ Open Babel не найден")
        print("Установите: conda install -c conda-forge openbabel")
        print("или: pip install openbabel-wheel\n")
        return False

if __name__ == "__main__":
    folder_path = "/Users/egorilin/Desktop/MSU_AI/xyz_number"
    
    # Проверяем Open Babel
    has_obabel = check_obabel()
    method = 'both' if has_obabel else 'rdkit'
    
    # Получаем словарь
    xyz_smiles_dict = process_xyz_folder(folder_path, method=method)
    
    # Сохраняем результаты
    if xyz_smiles_dict:
        output_file = "xyz_to_smiles_results.txt"
        with open(output_file, 'w') as f:
            for name in sorted(xyz_smiles_dict.keys(), key=lambda x: int(x) if x.isdigit() else x):
                f.write(f"{name}\t{xyz_smiles_dict[name]}\n")
        
        print(f"\n✓ Результаты сохранены в {output_file}")
        print(f"\nПервые 10 результатов:")
        for name in list(sorted(xyz_smiles_dict.keys(), key=lambda x: int(x) if x.isdigit() else x))[:10]:
            print(f"{name}: {xyz_smiles_dict[name]}")

✓ Open Babel установлен

Найдено 417 XYZ файлов
Начинаю обработку...



[00:08:44] Cannot convert '-3.3e-05' to double on line 27



Обработано 50/417 файлов... (Успешно: 50)
Обработано 100/417 файлов... (Успешно: 100)
Обработано 150/417 файлов... (Успешно: 150)
Обработано 200/417 файлов... (Успешно: 200)


[00:09:19] Unable to recognize the number of atoms: cannot convert 'O       -0.508584000     -1.218359000     -1.428310000' to unsigned int on line 0



Обработано 250/417 файлов... (Успешно: 249)
Обработано 300/417 файлов... (Успешно: 299)
Обработано 350/417 файлов... (Успешно: 349)
Обработано 400/417 файлов... (Успешно: 399)

Готово! Успешно: 416, Ошибок: 1

Список неудачных файлов сохранён в failed_files.txt

✓ Результаты сохранены в xyz_to_smiles_results.txt

Первые 10 результатов:
0: [H]OC(=O)c1c([H])c([H])c([H])c([H])c1-c1c2c([H])c(Br)c(=O)c(Br)c-2oc2c(Br)c(O[H])c(Br)c([H])c12
1: [H]OC(=O)c1c([H])c([H])c([H])c([H])c1-c1c2c([H])c([H])c(=O)c([H])c-2oc2c([H])c(O[H])c([H])c([H])c12
2: [H]OC(=O)c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2c([H])c(I)c(=O)c(I)c-2oc2c(I)c(O[H])c(I)c([H])c12
3: c1c(cc2c(c1)C(=C1[C](O2)C=C(C=C1)N(CC)CC)c1c(cccc1)C(=O)O)N(CC)CC
4: [H]c1c(C([H])([H])[H])c(C([H])([H])[H])c([H])c2c1nc1c(=O)n(C([H])([H])[H])c(=O)nc-1n2C([H])([H])[C@]([H])(OC(=O)C([H])([H])[H])[C@]([H])(OC(=O)C([H])([H])[H])[C@]([H])(OC(=O)C([H])([H])[H])C([H])([H])OC(=O)C([H])([H])[H]
5: [H]c1c([H])c(N(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c1C(=O)c1c([H])c

In [7]:
len(xyz_smiles_dict)

416

In [8]:
xyz_smiles_dict

{'0': '[H]OC(=O)c1c([H])c([H])c([H])c([H])c1-c1c2c([H])c(Br)c(=O)c(Br)c-2oc2c(Br)c(O[H])c(Br)c([H])c12',
 '1': '[H]OC(=O)c1c([H])c([H])c([H])c([H])c1-c1c2c([H])c([H])c(=O)c([H])c-2oc2c([H])c(O[H])c([H])c([H])c12',
 '10': 'c1ccc2c(c1)C(=C1C(=CC=C[CH]1)N2C)c1c(cc(cc1C)C)C',
 '100': 'c1cccc2[C]3C=CC=CN3[Ir]34([N@]([C](C=C(N3C3CCCCC3)C)C)C3CCCCC3)(c12)N1[C](c2ccccc42)C=CC=C1',
 '101': 'c1c(ccc2[C]3C=CC=CN3[Ir]34(N(C(=C[C](N3c3ccccc3)N(C)C)N(C)C)c3ccccc3)(c12)N1[C](c2ccc(cc42)C(C)(C)C)C=CC=C1)C(C)(C)C',
 '102': 'c1cccc2[C]3C=CC=CN3[Ir]34(N([C](C=C(N3c3ccccc3)C)C)c3ccccc3)(c12)N1[C](c2ccccc42)C=CC=C1',
 '103': 'c1cccc2[C]3C=CC=CN3[Ir]34(N(C(=C[C](N3c3ccccc3)N(C)C)N(C)C)c3ccccc3)(c12)N1[C](c2ccccc42)C=CC=C1',
 '104': 'c1cccc2[C]3C=CC=CN3[Ir]34(N([C](C=C(N3c3ccc(cc3)OC)C)C)c3ccc(cc3)OC)(c12)N1[C](c2ccccc42)C=CC=C1',
 '105': 'c1cccc2c1[Ir]13(N4C=C[CH]N24)(N([C](C=C(N1c1ccccc1)C)C)c1ccccc1)N1C=C[CH]N1c1ccccc31',
 '106': 'c1cccc2c1[Ir]13(N4C=C[CH]N24)(N([C](C=C(N1c1ccccc1)N(C)C)N(C)C)c1ccccc1)N1C